In [1]:
#General
import numpy as np
import pandas as pd
from scipy.stats import uniform
import os
from skimpy import skim
from tqdm import tqdm

#Manejo de texto plano
import re
import unicodedata

#Graficas
import matplotlib.pyplot as plt
import seaborn as sns

#Modelo de regresion logistica y Bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

#Busqueda de hiper-parametros, train test, metricas
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, roc_curve, confusion_matrix, make_scorer

#Modelo de clasificación
from sklearn.svm import SVC

### lectura de Archivo

In [3]:
path_file = "../data/Incendios_forestales.csv"

In [4]:
df = pd.read_csv(path_file)

In [5]:
df.head()

,anio,Clave_del_incendio,latitud,longitud,Clave_Municipio,Estado,Municipio,CVE_ENT,CVE_MUN,CVEGEO,...,Entidad,fn_Clave_del_incendio,fn_Predio,fn_Causa,fn_Causa_especifica,fn_Duracion_dias,fn_Tipo_de_incendio,fn_Tipo_Vegetacion,fn_Tipo_impacto,fn_Tamano
0,2022,22-01-0001,21.997253,-102.230097,11,Aguascalientes,San Francisco de los Romo,1,11,1101,...,Aguascalientes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,22-01-0002,21.694700,-102.265758,1,Aguascalientes,Aguascalientes,1,1,101,...,Aguascalientes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,22-01-0003,21.686344,-102.274769,1,Aguascalientes,Aguascalientes,1,1,101,...,Aguascalientes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,22-01-0004,21.837694,-102.247233,1,Aguascalientes,Aguascalientes,1,1,101,...,Aguascalientes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,22-01-0005,21.752181,-102.480086,1,Aguascalientes,Aguascalientes,1,1,101,...,Aguascalientes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 22332  │ │ float64     │ 17    │                                                          │
│ │ Number of columns │ 41     │ │ string      │ 16    │                                                          │
│ └───────────────────┴────────┘ │ int64       │ 8     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                    All null                                                     │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column                                                           ┃ NA                   ┃ NA %             ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩  │
│ │ fn_Predio                                                        │                22332 │              100 │  │
│ │ fn_Causa                                                         │                22332 │              100 │  │
│ │ fn_Causa_especifica                                              │                22332 │              100 │  │
│ │ fn_Duracion_dias                                                 │                22332 │              100 │  │
│ │ fn_Tipo_de_incendio                                              │                22332 │              100 │  │
│ │ fn_Tipo_Vegetacion                                               │                22332 │              100 │  │
│ │ fn_Tipo_impacto                                                  │                22332 │              100 │  │
│ │ fn_Tamano                                                        │                22332 │              100 │  │
│ └──────────────────────────────────────────────────────────────────┴──────────────────────┴──────────────────┘  │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column      ┃ NA    ┃ NA %        ┃ mean   ┃ sd     ┃ p0     ┃ p25    ┃ p50   ┃ p75    ┃ p100     ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩  │
│ │ anio        │     0 │           0 │   2023 │ 0.8099 │   2022 │   2022 │  2023 │   2024 │     2024 │ ▇  ▇ ▇ │  │
│ │ latitud     │     0 │           0 │  20.49 │  3.052 │  14.76 │  19.12 │ 19.57 │  20.81 │     32.6 │ ▁▇▂▁▁  │  │
│ │ longitud    │     0 │           0 │ -100.9 │  4.249 │ -117.1 │ -103.7 │  -100 │ -98.89 │   -86.84 │  ▁▅▇▁  │  │
│ │ Clave_Munic │     0 │           0 │     55 │  64.95 │      1 │     12 │    34 │     82 │      570 │   ▇▂   │  │
│ │ ipio        │       │             │        │        │        │        │       │        │          │        │  │
│ │ CVE_ENT     │     0 │           0 │  14.51 │  6.344 │      1 │      9 │    14 │     16 │       32 │ ▁▅▇▂▁▁ │  │
│ │ CVE_MUN     │     0 │           0 │     55 │  64.95 │      1 │     12 │    34 │     82 │      570 │   ▇▂   │  │
│ │ CVEGEO      │     0 │           0 │   4819 │   5646 │     11 │    909 │  2908 │   7216 │    56720 │   ▇▁   │  │
│ │ Arbolado_Ad │     0 │           0 │  3.772 │  44.82 